In [ ]:
!pip install -q transformers peft accelerate bitsandbytes pandas tqdm google-generativeai

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN').strip()
print(f"Token successfully loaded: {bool(hf_token)}")

login(token=hf_token)
# =========================================

# 🔐 SET YOUR GEMINI API KEY

# =========================================

GEMINI_API_KEY = userdata.get('GEMINI_TOKEN')
genai.configure(api_key=GEMINI_API_KEY)

judge_model = genai.GenerativeModel("gemini-2.5-flash")

# # =========================================

# # 📦 MODEL SETUP

# # =========================================

# base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
# adapter_id = "pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters"


# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# print("Loading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)

# print("Loading BASE model...")
# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     token=hf_token
# )

# print("Loading SFT model separately...")
# base_model_for_sft = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     token=hf_token
# )

# sft_model = PeftModel.from_pretrained(base_model_for_sft, adapter_id)

# =========================================
# 📦 MODEL SETUP (The Ultimate Showdown)
# =========================================

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)

# ---------------------------------------------------------
# CORNER 1: YOUR PHASE 1 SFT MODEL (Mapped to 'base_model')
# ---------------------------------------------------------
print("Loading Phase 1 SFT Model...")
base_model_for_phase1 = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
# We map this to 'base_model' so your existing loop works perfectly!
base_model = PeftModel.from_pretrained(
    base_model_for_phase1,
    "pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters"
)

# ---------------------------------------------------------
# CORNER 2: YOUR PHASE 2 DPO MODEL (Mapped to 'sft_model')
# ---------------------------------------------------------
print("Loading Phase 2 DPO Model separately...")
base_model_for_phase2 = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
# We map this to 'sft_model' so your existing loop works perfectly!
sft_model = PeftModel.from_pretrained(
    base_model_for_phase2,
    "pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters"
)

# =========================================

# 🧪 TEST PROMPTS (DPO-style)

# =========================================

prompts = [
    "Explain inflation in 2 lines.",
    "Explain stock market index simply.",
    "Give 3 bullet points on overfitting.",
    "What is machine learning in one paragraph?",
    "Explain bias vs variance clearly.",
    "Summarize: AI is transforming industries rapidly.",
    "Compare CNN vs RNN briefly.",
    "Explain gradient descent like I'm a beginner.",
    "Why does overfitting happen?",
    "Explain transformers in simple terms."
]


# =========================================

# 🤖 GENERATION FUNCTION

# =========================================

def generate(model, prompt):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


# =========================================

# ⚖️ GEMINI JUDGE FUNCTION

# =========================================

def judge(prompt, answer_a, answer_b):
    judge_prompt = f"""
    You are an expert evaluator for instruction-tuned LLMs.

    Dataset style: preference-based (like DPO).

    Evaluate the following answers:

    PROMPT:
    {prompt}

    ANSWER A:
    {answer_a}

    ANSWER B:
    {answer_b}

    Criteria:

    * correctness
    * relevance
    * clarity
    * conciseness
    * instruction-following

    Which answer is better?

    Reply ONLY with:
    A or B
    """

    response = judge_model.generate_content(judge_prompt)
    return response.text.strip()


# =========================================

# 🔁 EVALUATION LOOP

# =========================================

results = []

print("\n🚀 Running Evaluation...\n")

for prompt in tqdm(prompts):
    base_out = generate(base_model, prompt)
    sft_out = generate(sft_model, prompt)

    winner = judge(prompt, base_out, sft_out)

    results.append({
        "prompt": prompt,
        "base_output": base_out,
        "sft_output": sft_out,
        "winner": winner
    })


# =========================================

# 📊 RESULTS

# =========================================

df = pd.DataFrame(results)

sft_wins = (df["winner"] == "B").sum()
base_wins = (df["winner"] == "A").sum()

win_rate = sft_wins / len(df)

print("\n📊 FINAL RESULTS:")
print(f"SFT Wins: {sft_wins}")
print(f"Base Wins: {base_wins}")
print(f"Win Rate: {win_rate:.2f}")

# =========================================

# 💾 SAVE RESULTS

# =========================================

df.to_csv("evaluation_results.csv", index=False)

print("\n✅ Saved as evaluation_results.csv")

In [ ]:
# =========================================

# 🔥 IMPROVED EVALUATION (Score-Based + Better Prompts)

# =========================================

import torch
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

# =========================================

# 🔐 AUTH

# =========================================

hf_token = userdata.get('HF_TOKEN').strip()
login(token=hf_token)

GEMINI_API_KEY = userdata.get('GEMINI_TOKEN')
genai.configure(api_key=GEMINI_API_KEY)

judge_model = genai.GenerativeModel("gemini-2.5-flash")

# =========================================

# 📦 MODEL SETUP (3 MODELS)

# =========================================

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
load_in_4bit=True,
bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)

# -------- BASE MODEL --------

print("Loading BASE model...")
base_model = AutoModelForCausalLM.from_pretrained(
base_model_id,
quantization_config=bnb_config,
device_map="auto",
token=hf_token
)

# -------- SFT MODEL --------

print("Loading SFT model...")
base_for_sft = AutoModelForCausalLM.from_pretrained(
base_model_id,
quantization_config=bnb_config,
device_map="auto",
token=hf_token
)
sft_model = PeftModel.from_pretrained(
base_for_sft,
"pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters"
)

# -------- DPO MODEL --------

print("Loading DPO model...")
base_for_dpo = AutoModelForCausalLM.from_pretrained(
base_model_id,
quantization_config=bnb_config,
device_map="auto",
token=hf_token
)
dpo_model = PeftModel.from_pretrained(
base_for_dpo,
"pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters"
)

# =========================================

# 🧪 BETTER PROMPTS (Instruction Stress Test)

# =========================================

prompts = [
"Explain inflation in exactly 2 lines.",
"Give ONLY 3 bullet points on overfitting.",
"Explain stock market index in 1 simple sentence.",
"Explain machine learning in one paragraph (max 80 words).",
"Compare CNN vs RNN in 3 concise bullet points.",
"Explain gradient descent like I'm 10 years old (max 60 words).",
"Summarize in one sentence: AI is transforming industries rapidly.",
"Explain bias vs variance in exactly 3 points.",
"Why does overfitting happen? Answer in 2 lines only.",
"Explain transformers simply without examples (max 70 words)."
]
prompts = prompts[:5]

# =========================================

# 🤖 GENERATION

# =========================================

def generate(model, prompt):
  messages = [{"role": "user", "content": prompt}]


  inputs = tokenizer.apply_chat_template(
      messages,
      return_tensors="pt",
      add_generation_prompt=True,
      return_dict=True
  ).to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=120,
      do_sample=False
  )

  input_len = inputs["input_ids"].shape[1]
  return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


# =========================================

# ⚖️ SCORE-BASED JUDGE

# =========================================

def score_answer(prompt, answer):
  judge_prompt = f"""
  You are an expert evaluator of LLM responses.

  PROMPT:
  {prompt}

  ANSWER:
  {answer}

  Score from 1 to 10 on:

  1. Correctness
  2. Clarity
  3. Relevance
  4. Conciseness
  5. Instruction-following

  Return ONLY in this format:
  score: X
  """


  response = judge_model.generate_content(judge_prompt)

  try:
      score = float(response.text.strip().split(":")[-1])
  except:
      score = 5.0  # fallback

  return score


# =========================================

# 🔁 EVALUATION LOOP

# =========================================

results = []

print("\n🚀 Running Evaluation...\n")

for prompt in tqdm(prompts):
  base_out = generate(base_model, prompt)
  sft_out = generate(sft_model, prompt)
  dpo_out = generate(dpo_model, prompt)


  base_score = score_answer(prompt, base_out)
  sft_score = score_answer(prompt, sft_out)
  dpo_score = score_answer(prompt, dpo_out)

  results.append({
      "prompt": prompt,
      "base_score": base_score,
      "sft_score": sft_score,
      "dpo_score": dpo_score,
      "base_output": base_out,
      "sft_output": sft_out,
      "dpo_output": dpo_out
  })


# =========================================

# 📊 RESULTS

# =========================================

df = pd.DataFrame(results)

print("\n📊 AVERAGE SCORES:")
print(f"Base: {df['base_score'].mean():.2f}")
print(f"SFT : {df['sft_score'].mean():.2f}")
print(f"DPO : {df['dpo_score'].mean():.2f}")

# =========================================

# 🏆 PAIRWISE WIN COUNTS

# =========================================

def win_rate(a, b):
    return (df[a] > df[b]).sum() / len(df)

print("\n🏆 WIN RATES:")
print(f"SFT > Base : {win_rate('sft_score','base_score'):.2f}")
print(f"DPO > Base : {win_rate('dpo_score','base_score'):.2f}")
print(f"DPO > SFT  : {win_rate('dpo_score','sft_score'):.2f}")

# =========================================

# 💾 SAVE

# =========================================

df.to_csv("evaluation_scores.csv", index=False)
print("\n✅ Saved as evaluation_scores.csv")

In [ ]:
!pip install "transformers<5.0" accelerate bitsandbytes peft lm-eval

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN').strip()
login(token=hf_token)

In [ ]:
%%bash
lm_eval \
  --model hf \
  --model_args pretrained=meta-llama/Llama-3.2-1B-Instruct,load_in_4bit=True \
  --tasks truthfulqa_mc1,arc_easy \
  --device cuda:0 \
  --batch_size 4 \
  --output_path ./results/base \
  --log_samples

In [ ]:
%%bash
lm_eval \
  --model hf \
  --model_args pretrained=meta-llama/Llama-3.2-1B-Instruct,peft=pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters,load_in_4bit=True \
  --tasks truthfulqa_mc1,arc_easy \
  --device cuda:0 \
  --batch_size 4 \
  --output_path ./results/sft \
  --log_samples

In [ ]:
%%bash
lm_eval \
  --model hf \
  --model_args pretrained=meta-llama/Llama-3.2-1B-Instruct,peft=pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters,load_in_4bit=True \
  --tasks truthfulqa_mc1,arc_easy \
  --device cuda:0 \
  --batch_size 4 \
  --output_path ./results/dpo \
  --log_samples